In [9]:
!pip install catboost


   ---------------------------------------- 0.0/102.4 MB ? eta -:--:--
   ---------------------------------------- 0.5/102.4 MB 2.6 MB/s eta 0:00:39
    --------------------------------------- 1.3/102.4 MB 3.7 MB/s eta 0:00:28
    --------------------------------------- 1.8/102.4 MB 3.6 MB/s eta 0:00:28
   - -------------------------------------- 2.9/102.4 MB 3.6 MB/s eta 0:00:28
   - -------------------------------------- 3.7/102.4 MB 3.6 MB/s eta 0:00:28
   - -------------------------------------- 4.2/102.4 MB 3.6 MB/s eta 0:00:28
   - -------------------------------------- 5.0/102.4 MB 3.6 MB/s eta 0:00:27
   -- ------------------------------------- 5.8/102.4 MB 3.6 MB/s eta 0:00:27
   -- ------------------------------------- 6.6/102.4 MB 3.6 MB/s eta 0:00:27
   -- ------------------------------------- 7.3/102.4 MB 3.6 MB/s eta 0:00:27
   --- ------------------------------------ 8.1/102.4 MB 3.6 MB/s eta 0:00:26
   --- ------------------------------------ 8.9/102.4 MB 3.6 MB/s eta 0


[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [38]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [39]:
df_train = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')

In [40]:
df_train

,event_id,num_perimeters_0_5h,dt_first_last_0_5h,low_temporal_resolution_0_5h,area_first_ha,area_growth_abs_0_5h,area_growth_rel_0_5h,area_growth_rate_ha_per_h,log1p_area_first,log1p_growth,...,dist_fit_r2_0_5h,alignment_cos,alignment_abs,cross_track_component,along_track_speed,event_start_hour,event_start_dayofweek,event_start_month,time_to_hit_hours,event
0,10892457,3,4.265188,0,79.696304,2.875935,0.036086,0.674281,4.390693,1.354787,...,0.886373,-0.054649,0.054649,-1.937219,-0.106026,19,4,5,18.892512,0
1,11757157,2,1.169918,0,8.946749,0.000000,0.000000,0.000000,2.297246,0.000000,...,0.000000,-0.568898,0.568898,-0.000000,-0.000000,4,4,6,22.048108,1
2,11945086,4,4.777526,0,106.482638,0.000000,0.000000,0.000000,4.677329,0.000000,...,0.000000,0.882385,0.882385,0.000000,0.000000,22,4,8,0.888895,1
3,12044083,1,0.000000,1,67.631125,0.000000,0.000000,0.000000,4.228746,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,20,5,8,60.953021,0
4,12052347,2,4.975273,0,35.632874,0.000000,0.000000,0.000000,3.600946,0.000000,...,0.000000,0.934634,0.934634,-0.000000,0.000000,21,5,7,44.990274,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
216,97075632,1,0.000000,1,51.295195,0.000000,0.000000,0.000000,3.956904,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,17,6,6,66.340624,0
217,97362560,2,1.127102,0,1.176991,0.000000,0.000000,0.000000,0.777943,0.000000,...,0.000000,-0.277779,0.277779,-0.000000,-0.000000,18,1,7,5.694898,1
218,97805715,2,3.710653,0,71.946930,0.000000,0.000000,0.000000,4.289732,0.000000,...,0.000000,0.694609,0.694609,0.000000,0.000000,18,1,9,44.011253,0
219,99071478,1,0.000000,1,20.223659,0.000000,0.000000,0.000000,3.055117,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,15,0,8,22.975783,1


In [41]:
df_test

,event_id,num_perimeters_0_5h,dt_first_last_0_5h,low_temporal_resolution_0_5h,area_first_ha,area_growth_abs_0_5h,area_growth_rel_0_5h,area_growth_rate_ha_per_h,log1p_area_first,log1p_growth,...,projected_advance_m,dist_accel_m_per_h2,dist_fit_r2_0_5h,alignment_cos,alignment_abs,cross_track_component,along_track_speed,event_start_hour,event_start_dayofweek,event_start_month
0,10662602,1,0.000000,1,2.452217,0.000000,0.00000,0.000000,1.239017,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,3,7
1,13353600,1,0.000000,1,131.669588,0.000000,0.00000,0.000000,4.887862,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,22,0,8
2,13942327,1,0.000000,1,6.723104,0.000000,0.00000,0.000000,2.044216,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2,6,7
3,16112781,1,0.000000,1,285.416736,0.000000,0.00000,0.000000,5.657448,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,1,7
4,17132808,7,3.459331,0,61.098604,12.516633,0.20486,3.618224,4.128724,2.603921,...,13.54413,-22.687575,0.044572,0.158550,0.158550,-24.414806,3.920562,23,5,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
90,94627327,2,4.614923,0,24.438326,0.000000,0.00000,0.000000,3.236257,0.000000,...,0.00000,0.000000,0.000000,0.763809,0.763809,0.000000,0.000000,22,0,6
91,96570675,1,0.000000,1,155.843418,0.000000,0.00000,0.000000,5.055248,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,21,2,7
92,97225766,1,0.000000,1,87.761553,0.000000,0.00000,0.000000,4.485954,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,4,1,7
93,98446281,2,4.335964,0,11.391108,0.000000,0.00000,0.000000,2.516979,0.000000,...,0.00000,0.000000,0.000000,-0.305683,0.305683,0.000000,-0.000000,17,0,9


In [42]:
test_ids = df_test['event_id'].copy()

In [43]:
drop_cols = ['event', 'time_to_hit_hours', 'event_id']
X_train = df_train.drop(columns=drop_cols)
y_train = df_train['event']

X_test = df_test.drop(columns=drop_cols, errors='ignore')

In [23]:
from catboost import CatBoostClassifier, Pool

In [83]:
model = CatBoostClassifier(
     iterations=2500,
    depth=9,
    learning_rate=0.025,
    loss_function='Logloss',
    eval_metric='AUC',
    l2_leaf_reg=8,
    subsample=0.8, 
    bootstrap_type='Bernoulli',
    verbose=300,
    random_seed=42
)
model.fit(X_train,y_train)

0:	total: 2.54ms	remaining: 1.26s
100:	total: 135ms	remaining: 533ms
200:	total: 248ms	remaining: 369ms
300:	total: 357ms	remaining: 236ms
400:	total: 471ms	remaining: 116ms
499:	total: 581ms	remaining: 0us


In [84]:
model.get_best_iteration()
model.get_best_score()


{'learn': {'Logloss': 0.0009685837549072631}}

In [85]:
base_prob = model.predict_proba(X_test)[:, 1]


In [86]:
df_sub = pd.read_csv('sample_submission.csv')

In [87]:
print(len(X_train), len(y_train))  
print(len(X_test), len(test_ids))  
print(len(base_prob))              


221 221
95 95
95


In [88]:
prob_12h = np.clip(base_prob * 0.6, 0, 1)
prob_24h = np.clip(base_prob * 0.8, 0, 1)
prob_48h = np.clip(base_prob * 1.0, 0, 1)
prob_72h = np.clip(base_prob * 1.2, 0, 1)

prob_24h = np.maximum(prob_24h, prob_12h)
prob_48h = np.maximum(prob_48h, prob_24h)
prob_72h = np.maximum(prob_72h, prob_48h)


In [89]:
df_sub = pd.DataFrame({
    'event_id': test_ids,
    'prob_12h': prob_12h,
    'prob_24h': prob_24h,
    'prob_48h': prob_48h,
    'prob_72h': prob_72h
})


In [90]:
df_sub.to_csv('CatBoost_ov2.csv', index=False)


In [91]:
df_sub.head()


,event_id,prob_12h,prob_24h,prob_48h,prob_72h
0,10662602,0.000102,0.000136,0.000170,0.000204
1,13353600,0.599699,0.799599,0.999498,1.000000
2,13942327,0.000103,0.000137,0.000171,0.000206
3,16112781,0.599680,0.799573,0.999466,1.000000
4,17132808,0.014066,0.018754,0.023443,0.028131
